# Distribution Deep Drive

## Import Libraries

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector
from dotenv import load_dotenv
import os

## Load Environment Variables and Create Charts Folder

In [2]:
load_dotenv()

os.makedirs("charts", exist_ok=True)
# Optional styling
sns.set_style("whitegrid")

## Connect to MySQL Database

In [3]:
try:
    
    conn = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database="weather2_db"
    )

    cursor = conn.cursor()

    print("MySQL Database Connected Successfully")

except Exception as e:
    
    print("Database Connection Error:", e)

MySQL Database Connected Successfully


## Create Weather Table

In [4]:
try:
    
    create_table_query = """
    CREATE TABLE IF NOT EXISTS weather_data (
        id INT AUTO_INCREMENT PRIMARY KEY,
        city VARCHAR(100),
        date DATE,
        max_temp FLOAT,
        min_temp FLOAT,
        rainfall FLOAT
    )
    """

    cursor.execute(create_table_query)

    conn.commit()

    print("Table Created Successfully")

except Exception as e:
    
    print("Table Creation Error:", e)

Table Created Successfully


## Nepal Cities

In [5]:
cities = {
    "Kathmandu": (27.7172, 85.3240),
    "Pokhara": (28.2096, 83.9856),
    "Biratnagar": (26.4525, 87.2718),
    "Nepalgunj": (28.0500, 81.6167),
    "Dharan": (26.8121, 87.2834)
}

## Proper API Fetching

In [6]:
for city, coords in cities.items():

    try:

        lat, lon = coords

        url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}"
            f"&daily=temperature_2m_max,"
            f"temperature_2m_min,"
            f"precipitation_sum"
            f"&forecast_days=7"
            f"&timezone=auto"
        )

        response = requests.get(url, timeout=10)

        # CHECK STATUS
        if response.status_code != 200:
            print(f"Failed to fetch data for {city}")
            continue

        data = response.json()

        # VALIDATE RESPONSE
        if "daily" not in data:
            print(f"Daily data missing for {city}")
            continue

        dates = data["daily"]["time"]
        max_temps = data["daily"]["temperature_2m_max"]
        min_temps = data["daily"]["temperature_2m_min"]
        rainfall = data["daily"]["precipitation_sum"]

        for d, max_t, min_t, rain in zip(
            dates,
            max_temps,
            min_temps,
            rainfall
        ):

            insert_query = """
            INSERT INTO weather_data
            (city, date, max_temp, min_temp, rainfall)
            VALUES (%s, %s, %s, %s, %s)
            """

            values = (
                city,
                d,
                max_t,
                min_t,
                rain
            )

            cursor.execute(insert_query, values)

        conn.commit()

        print(f"{city} data inserted successfully")

    except requests.exceptions.RequestException as e:

        print(f"Request Error for {city}: {e}")

    except Exception as e:

        print(f"General Error for {city}: {e}")

Kathmandu data inserted successfully
Pokhara data inserted successfully
Biratnagar data inserted successfully
Nepalgunj data inserted successfully
Dharan data inserted successfully


## Load Data into Pandas

In [7]:
try:

    query = "SELECT * FROM weather_data"

    df = pd.read_sql(query, conn)

    print("Data Loaded Successfully")

    df.head()

except Exception as e:

    print("Error Loading Data:", e)

Data Loaded Successfully


C:\Users\sumit\AppData\Local\Temp\ipykernel_14532\2527721272.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## Shape, Info, Missing Values

In [8]:
print(df.shape)

df.info()

df.isnull().sum()

(35, 6)
<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        35 non-null     int64  
 1   city      35 non-null     str    
 2   date      35 non-null     object 
 3   max_temp  35 non-null     float64
 4   min_temp  35 non-null     float64
 5   rainfall  35 non-null     float64
dtypes: float64(3), int64(1), object(1), str(1)
memory usage: 1.8+ KB


id          0
city        0
date        0
max_temp    0
min_temp    0
rainfall    0
dtype: int64

## Statistical Summary

In [12]:
df.describe()


,id,max_temp,min_temp,rainfall
count,35.000000,35.000000,35.00000,35.000000
mean,18.000000,31.205714,22.22000,3.042857
std,10.246951,4.041472,2.89998,2.899826
min,1.000000,25.600000,16.90000,0.000000
25%,9.500000,28.450000,19.50000,0.250000
50%,18.000000,30.100000,22.90000,2.500000
75%,26.500000,32.900000,24.60000,5.150000
max,35.000000,39.600000,27.40000,9.600000


In [13]:
print(df.head())

   id       city        date  max_temp  min_temp  rainfall
0   1  Kathmandu  2026-05-17      25.6      16.9       2.1
1   2  Kathmandu  2026-05-18      26.9      19.0       3.0
2   3  Kathmandu  2026-05-19      27.1      19.3       2.5
3   4  Kathmandu  2026-05-20      26.2      18.2       2.7
4   5  Kathmandu  2026-05-21      26.0      18.5       3.6
